# Proof: Daily Losses, EU report against the site report

**Question.** If both reports use the same configuration, do they hold the same data?

**How.** We do not rebuild anything. We read the table **as Power BI loaded it** from each real
`.pbix` file, give the site table the one extra rule the EU report has (its machine filter), and
compare the two tables row by row.

Run one cell at a time. Each cell says what you should see. Set `SITE` in the first cell, run
everything, then change `SITE` and run again.

## Step 0. Get the data out of the .pbix files (once)

The helper `scripts/helper-pbix-extract.py` reads the loaded tables of the three files and saves
them in `downloads/pbix_data/` (git-ignored, customer data). Run it once in a terminal:

```
python scripts/helper-pbix-extract.py .
```

It needs `pip install pbixray`. Skip this step if `downloads/pbix_data/` already has the files.

## Step 1. Settings

In [ ]:
import pandas as pd

SITE = 'Hull'          # 'Hull' or 'Nottingham'
DATA = '../downloads/pbix_data/'

# The EU file was refreshed earlier than the site files: its data ends on 16 Sep 2026.
# We compare only the days both files hold completely.
END = '2026-09-16'     # compare days before this date

EU_TABLE = {'Hull': 'eu__Daily_Line_Losses_H', 'Nottingham': 'eu__Daily_Line_Losses_N'}[SITE]
SITE_TABLE = {'Hull': 'hull__New_Daily_Line_Losses', 'Nottingham': 'nott__Daily_Line_Losses_N'}[SITE]
print('EU report table  :', {'Hull': 'Daily Line Losses H', 'Nottingham': 'Daily Line Losses N'}[SITE], '(the EU query for this site, before the union)')
print('Site report table:', {'Hull': 'New Daily Line Losses', 'Nottingham': 'Daily Line Losses N'}[SITE])

## Step 2. Load both tables
You should see two row counts and two date ranges.

In [ ]:
eu = pd.read_parquet(DATA + EU_TABLE + '.parquet')
site_tbl = pd.read_parquet(DATA + SITE_TABLE + '.parquet')

for name, df in [('EU', eu), ('Site', site_tbl)]:
    print(f"{name:5} {len(df):>8,} rows   {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d}")

## Step 3. Keep only the days both files cover
The site files were refreshed a week later, so they hold more days. After this cell both tables
end on the same day.

In [ ]:
eu = eu[eu['Date'] < END].copy()
site_tbl = site_tbl[site_tbl['Date'] < END].copy()
print(f'EU   {len(eu):>8,} rows')
print(f'Site {len(site_tbl):>8,} rows   <- more rows: the site report keeps machines the EU drops')

## Step 4. Turn time columns into minutes
Power BI stores durations as a time of day (for example 08:30:00). We turn them into minutes so
they can be summed. Number columns stay as they are.

In [ ]:
BASE_COLS = ['Duration', 'Loss Units', 'Time in Mins', 'Volume Total']
TARGET_COLS = ['Value']

def to_minutes(s):
    if pd.api.types.is_datetime64_any_dtype(s):
        return ((s - pd.Timestamp('1899-12-30')).dt.total_seconds() / 60).round(4)
    return pd.to_numeric(s).astype(float)

for df in (eu, site_tbl):
    for col in BASE_COLS + TARGET_COLS:
        df[col] = to_minutes(df[col])
print('done')

## Step 5. Give the site table the same configuration as the EU report
The EU query has one extra step at the end: a machine filter. Hull: drop 19 process machines. Nottingham: drop 9 machines (Bosch, Fette*, Gran0, S14).
We apply that same filter to the site table. You should see the machines it removes.

(We trim spaces from the machine name: the EU list has `"Bosch Capsulator "` with a trailing
space, Power BI shows it without.)

In [ ]:
HULL_EU_OUT = ['4 Tonne', '6 Tonne', '6000L', 'Analgesics Processing', 'Analgesics X Press',
               'Bosch Capsulator', 'Dettol 1', 'Evolution', 'Fitten', 'Gaviscon TPR',
               'Gaviscon XP700(A)', 'GG4', 'GG5', 'GG6', 'GG7', 'Lemsip Process',
               'Senokot Press', 'Symex suite', 'Vacumix']
NOTT_TU_KEEP = ['N1', 'N10', 'N11', 'N12', 'N3', 'N4', 'N5', 'N6', 'N7', 'N8', 'N9 Auto',
                'O5', 'O6', 'S04', 'S05', 'S06', 'S08', 'S10', 'S11', 'S12', 'S13', 'T2', 'T7']
NOTT_LOSS_OUT = ['Bosch', 'Fette 23', 'Fette12', 'Fette2', 'Fette20', 'Fette4', 'Fette5',
                 'Gran0', 'S14']

machine = site_tbl['MachineName'].str.strip()
if SITE == 'Hull':
    keep = ~machine.isin(HULL_EU_OUT)
else:
    keep = ~machine.isin(NOTT_LOSS_OUT)

removed = site_tbl[~keep]
site_cfg = site_tbl[keep].copy()
print(f'Removed {len(removed):,} rows. Machines removed:')
print(removed['MachineName'].value_counts().to_string())
print()
print(f'EU                  {len(eu):>8,} rows')
print(f'Site, EU configured {len(site_cfg):>8,} rows   <- now (almost) the same')

## Step 6. Match the rows one by one
Key = machine + stop start time (the column "Merged"). Every key should be in both tables.

In [ ]:
KEYS = ['Merged']

a = eu.groupby(KEYS, dropna=False).size().rename('rows_eu')
b = site_cfg.groupby(KEYS, dropna=False).size().rename('rows_site')
match = pd.concat([a, b], axis=1)

print(f"keys in both tables: {match.dropna().shape[0]:>8,}")
print(f"keys only in EU    : {match['rows_site'].isna().sum():>8,}")
print(f"keys only in site  : {match['rows_eu'].isna().sum():>8,}")
only = match[match.isna().any(axis=1)].reset_index()
if len(only):
    print()
    print('The keys that are on one side only (look at the dates):')
    display(only.head(20))

## Step 7. Compare every measure on the matched rows
For each measure: the EU total, the site total, and how many keys differ.
**Expected: 0 for the measures that come straight from the exports.** The targets (`UPDT`, `PDT`,
`OEE Target`, `Value`) come from a glidepath file and are checked in step 9.

In [ ]:
sa = eu.groupby(KEYS, dropna=False)[BASE_COLS + TARGET_COLS].sum()
sb = site_cfg.groupby(KEYS, dropna=False)[BASE_COLS + TARGET_COLS].sum()
both = sa.join(sb, how='inner', lsuffix=' EU', rsuffix=' site')

rows = []
for col in BASE_COLS + TARGET_COLS:
    diff = both[col + ' EU'].fillna(0) - both[col + ' site'].fillna(0)
    rows.append({'measure': col,
                 'EU total': both[col + ' EU'].sum(),
                 'site total': both[col + ' site'].sum(),
                 'keys that differ': int((diff.abs() > 0.001).sum())})
result = pd.DataFrame(rows).set_index('measure')
pd.options.display.float_format = '{:,.2f}'.format
display(result)

## Step 8. Where are the few differences in the export measures?
If the export measures differ on a handful of keys, look at the dates. Differences only in the last
days before `END` mean the source file was updated between the two refreshes (the EU file was
refreshed on an earlier day). That is timing, not configuration.

In [ ]:
for col in BASE_COLS:
    diff = both[col + ' EU'].fillna(0) - both[col + ' site'].fillna(0)
    bad = diff[diff.abs() > 0.001]
    if len(bad) == 0:
        print(f'{col:18} no difference')
        continue
    keys = bad.reset_index()[KEYS]
    days = eu.merge(keys, on=KEYS)['Date'].dt.strftime('%Y-%m-%d').unique()
    print(f'{col:18} {len(bad)} keys differ, on days: {sorted(days)}')

## Step 9. The targets: a different configuration, on purpose
The targets are looked up in a glidepath file. The two reports use **different files and keys**:
- EU: the regional glidepath, key = machine-site + month (+ loss sub group for `Value`).
- Site: the site's own glidepath file, key = machine + month.
- Nottingham site file keeps the downtime targets as fractions (0.147), the regional file as percent
  (14.7): that is a factor of 100.
- Hull losses: the EU renames the sub groups (Minor Stop to Minor Stops ...) before the lookup.

So the targets differ because the configuration differs, not because the data differs. The table
shows, per target, how many keys differ and the ratio of the totals.

In [ ]:
for col in TARGET_COLS:
    e, s = both[col + ' EU'].sum(), both[col + ' site'].sum()
    n = int(((both[col + ' EU'].fillna(0) - both[col + ' site'].fillna(0)).abs() > 0.001).sum())
    ratio = e / s if s else float('nan')
    print(f'{col:12} EU {e:>14,.1f}  site {s:>14,.1f}  ratio {ratio:6.2f}  keys differ {n:,}')
if not TARGET_COLS:
    print('This table has no targets.')

## Step 10. One more EU step: the union into `Daily Losses EU`
The visuals of the EU report read `Daily Losses EU`, the union of all sites. While building it
the EU query reads the times as hours:minutes and **drops the seconds**. Rows and Good Parts stay
the same, minutes go down a little.

In [ ]:
union = pd.read_parquet(DATA + 'eu__Daily_Losses_EU' + '.parquet')
union = union[(union['Date'] < END) & (union['SiteName'] == SITE)]
for col in ['Duration']:
    before = eu[col].sum()
    after = pd.to_numeric(union[col]).astype(float).sum()
    print(f'{col:18} site part {before:>16,.2f}  union {after:>16,.2f}  difference {after - before:>12,.2f}')
print(f'rows: site part {len(eu):,}  union {len(union):,}')

## Step 11. Verdict
Read the printout: it is the answer for this table and site.

In [ ]:
export_ok = (result.loc[BASE_COLS, 'keys that differ'] == 0)
n_only = int(match.isna().any(axis=1).sum())
print(f'Rows: {len(eu):,} EU against {len(site_cfg):,} site (EU configured), {n_only} keys on one side only.')
for col, ok in export_ok.items():
    print(f"  {col:18} {'IDENTICAL' if ok else 'differs: see step 8 (dates)'}")
print('Targets: differ by design (step 9).' if TARGET_COLS else '')
print()
print('Conclusion: with the EU machine filter, the site report holds the same rows and the same')
print('export values as the EU report. Remaining differences are the refresh date (step 8), the')
print('glidepath configuration (step 9) and the seconds the EU union drops (step 10).')